# EXT 모델카드 — 제출 전 확인

`EXT_최종스펙.md` §10의 🔴 항목을 전부 코드로 확인한다. **이 노트북은 자립한다** — 다른 노트북 불필요.

| 확인 항목 | 셀 | 필요 |
|---|---|---|
| 라이브러리 버전 · GPU | §1 | — |
| 모델 revision 해시 고정 | §2 | 네트워크 |
| 🔴 **1단 OWLv2 지연 측정** | §4 | GPU · 평가셋 |
| 2단 Qwen 지연 재측정 | §6 | GPU 16GB · 평가셋 |
| 🔴 **골든 fixture 20장 + 스냅샷** | §7 | GPU · 평가셋 |
| fixture 회귀 검증 (평가 경로) | §8 | §7 이후 |
| 🔴 **fixture 회귀 (배포 경로)** | **§10** | **GPU · 원본 20장** |
| 모델카드 붙여넣기용 요약 | §9 | — |

## 실행 순서

```
필수 선행:  §0  →  §1  →  §3               (§3에서 OWLv2를 올린다, ~3분)
지연·픽스처: §4  →  §5  →  §7  →  §8        (~10분)
2단 지연:   §6                              (Qwen 로드 ~5분 + 측정 1분, 선택)
revision:  §2                              (★모델을 받은 뒤에 돌려야 로컬 해시가 찍힌다)
배포 회귀:  §10                             (GPU · ~2분, 다른 셀과 독립)
마지막:     §9                              (즉시)
```

🔴 **§2는 §3(·§6) 뒤에 돌릴 것.** 먼저 돌리면 원격 sha만 나오고 로컬 스냅샷이 비어 있다 —
모델카드에 적어야 하는 건 **실제로 받은 로컬 해시**다.

**§6(Qwen)은 선택이다.** 2단 지연은 이미 1.1초/호출로 실측돼 있고, 재확인이 목적이다.
GPU 메모리가 빠듯하면 §6을 건너뛰어도 §7~§9는 돌아간다.

⚠️ 산출물은 전부 `{OUT}/modelcard/`에 쌓인다. 다시 돌려도 덮어쓰므로 안전하다.

In [ ]:
# == §0 셋업 — 경로·평가셋 확인 ==
# 이 노트북은 자립한다. 다른 노트북을 안 돌려도 된다.
# 필요한 것: Drive의 평가셋(images/ · labels_*.json). §7 픽스처가 그 이미지를 쓴다.
import os, sys, json, platform, subprocess
from pathlib import Path

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass

OUT = next((c for c in (Path('/content/drive/MyDrive/kt_out/ext_evalset'),
                        Path('/content/drive/MyDrive/ext_evalset'),
                        Path('/content/ext_evalset'))
            if (c / 'images').is_dir()), None)
assert OUT, '★평가셋 폴더를 못 찾았다 — evalset 노트북 §A를 먼저 돌릴 것'
IMGDIR = OUT / 'images'
MC = OUT / 'modelcard'; MC.mkdir(exist_ok=True)

GT = {}
for f in sorted(OUT.glob('labels_*.json')):
    if f.name == 'labels_cap.json': continue
    for k, v in json.loads(f.read_text(encoding='utf-8')).items(): GT[k] = v
IMGS = sorted(p.name for p in IMGDIR.glob('*.jpg'))

print(f'평가셋   {OUT}')
print(f'이미지   {len(IMGS)}장')
print(f'사람라벨 {len(GT)}장 · 점 {sum(len(v["points"]) for v in GT.values())}개 '
      f'· 무결함 {sum(1 for v in GT.values() if not v["points"])}장')
print(f'산출물   {MC}')
assert IMGS, '★이미지가 없다'

In [ ]:
# == §1 라이브러리 버전 · GPU — 모델카드 "환경" 항목 ==
# 🔴 `transformers`는 **반드시** 고정할 것. OWLv2 후처리 API 이름이 버전 간 다르다
#    (`post_process_grounded_object_detection` ↔ `post_process_object_detection`).
import importlib.metadata as _md

PKGS = ['transformers', 'tokenizers', 'huggingface-hub', 'accelerate', 'safetensors',
        'qwen-vl-utils', 'torch', 'torchvision', 'pillow', 'numpy', 'scipy',
        'ultralytics', 'opencv-python']
ENV = {'python': platform.python_version(), 'platform': platform.platform()}
for p in PKGS:
    try: ENV[p] = _md.version(p)
    except Exception: ENV[p] = None

try:
    import torch
    ENV['torch.version.cuda'] = torch.version.cuda
    ENV['cudnn'] = torch.backends.cudnn.version()
    ENV['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    ENV['gpu_mem_GB'] = (round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                         if torch.cuda.is_available() else None)
except Exception as e:
    ENV['torch_error'] = str(e)
try:
    ENV['nvidia_driver'] = subprocess.run(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        capture_output=True, text=True).stdout.strip() or None
except Exception: ENV['nvidia_driver'] = None

(MC / 'env.json').write_text(json.dumps(ENV, ensure_ascii=False, indent=1), encoding='utf-8')
w = max(len(k) for k in ENV)
for k, v in ENV.items():
    print(f'{k:{w}s}  {v}' + ('   🔴 미설치' if v is None and k in PKGS else ''))
print(f'\n저장 {MC/"env.json"}')
if not ENV.get('transformers'):
    print('🔴 transformers가 없다 — §3에서 설치된다. 그 뒤 이 셀을 다시 돌려 버전을 기록할 것')

In [ ]:
# == §2 모델 revision 해시 고정 — 모델카드 "모델 식별" 항목 ==
# 🔴 `model_id`만 적으면 재현이 안 된다. HF 레포는 갱신되므로 **커밋 해시**를 박아야 한다.
#    아래가 찍어주는 sha를 그대로 모델카드와 코드(`from_pretrained(..., revision=...)`)에 넣는다.
OWL_ID = 'google/owlv2-large-patch14-ensemble'
VLM_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

try:
    from huggingface_hub import HfApi
except Exception:
    !pip -q install -U huggingface-hub
    from huggingface_hub import HfApi

api = HfApi()
REV = {}
for rid in (OWL_ID, VLM_ID):
    try:
        i = api.model_info(rid)
        REV[rid] = {'sha': i.sha,
                    'lastModified': str(getattr(i, 'lastModified', '')),
                    'files': len(getattr(i, 'siblings', []) or [])}
        print(f'{rid}\n   sha          {i.sha}\n   lastModified {REV[rid]["lastModified"]}')
    except Exception as e:
        REV[rid] = {'error': str(e)}
        print(f'{rid}\n   🔴 조회 실패: {e}')

# 로컬 캐시에 실제로 받아진 스냅샷 — 위 sha와 다르면 **코드가 쓰는 건 이쪽**이다
from huggingface_hub import constants as _hc
CACHE = Path(getattr(_hc, 'HF_HUB_CACHE', Path.home() / '.cache/huggingface/hub'))
print(f'\n로컬 캐시 {CACHE}')
for rid in (OWL_ID, VLM_ID):
    d = CACHE / ('models--' + rid.replace('/', '--')) / 'snapshots'
    got = sorted(p.name for p in d.iterdir()) if d.is_dir() else []
    REV.setdefault(rid, {})['local_snapshots'] = got
    print(f'   {rid}: {got or "(아직 안 받음)"}' +
          ('   ✅ 일치' if got and REV[rid].get('sha') in got else
           ('   ⚠️ 원격 sha와 다름 — 코드가 쓰는 건 로컬 쪽이다' if got else '')))

(MC / 'revisions.json').write_text(json.dumps(REV, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'\n저장 {MC/"revisions.json"}')
print('\n▶ 모델카드에는 **로컬 스냅샷 해시**를 적고, 코드에도 revision=으로 박을 것.')
if not any(REV.get(r, {}).get('local_snapshots') for r in (OWL_ID, VLM_ID)):
    print('\n🔴 로컬 스냅샷이 비었다 = 아직 모델을 안 받았다.')
    print('   **§3(·§6)을 돌린 뒤 이 셀을 한 번 더 실행**하면 실제로 받은 해시가 찍힌다.')
    print('   그때 원격 sha와 같은지 확인할 것 — 다르면 코드가 쓰는 건 로컬 쪽이다.')

In [ ]:
# == §3 OWLv2 로드 + detect (자립판) — §4·§7의 선행 ==
# unsup §1과 **같은 설정**이다. 이 노트북만으로 재현되도록 복제해 둔다.
# 🔴 OWLv2 전처리는 이미지를 정사각 패딩한다 → 직사각을 그대로 넣으면 좌표가 조용히 어긋난다.
#    우리가 먼저 정사각 패딩(원본 좌상단)해 내부 패딩을 무연산으로 만든다.
!pip -q install -U transformers accelerate

import torch, numpy as np, time
from PIL import Image
from transformers import Owlv2Processor, Owlv2ForObjectDetection

REVISION = None          # ★§2에서 확인한 해시를 여기 넣으면 완전 고정된다
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
_kw = {'revision': REVISION} if REVISION else {}
proc_d = Owlv2Processor.from_pretrained(OWL_ID, **_kw)
det = Owlv2ForObjectDetection.from_pretrained(OWL_ID, **_kw).to(DEV).eval()

QUERY_MAP = {
    '녹·부식':      ['rust', 'orange brown discoloration', 'brown stain on surface',
                     'oxidation on surface', 'reddish brown spot'],
    '벗겨짐·박리':  ['peeling plastic wrap', 'peeled off label',
                     'exposed metal under wrapper', 'wrapper coming off', 'peeling film edge'],
    '파손·찢김':    ['tear in plastic film', 'crack on surface', 'puncture hole', 'dent'],
    '긁힘·스크래치': ['scratch', 'scratch mark', 'scratched metal surface', 'scuff mark'],
    '들뜸':         ['air bubble under film', 'wrinkled plastic wrap', 'lifted film edge'],
    '오염·이물질':  ['dirt stain', 'smudge', 'white residue', 'foreign particle on surface'],
}
NEG_MAP = {
    '정상:인쇄·각인': ['printed text', 'printed letters on label', 'engraved serial number'],
    '정상:반사':      ['light reflection', 'specular highlight'],
    '정상:금속캡':    ['metal cap', 'battery terminal'],
    '정상:테두리':    ['bottom edge of cylinder', 'rim of metal can', 'silhouette edge'],
}
DROP_TAGS = {'정상:인쇄·각인', '정상:반사', '정상:테두리'}   # 🔴 금속캡은 절대 안 버린다
QUERIES = [q for v in QUERY_MAP.values() for q in v] + [q for v in NEG_MAP.values() for q in v]
TAG_OF  = {q: k for k, v in {**QUERY_MAP, **NEG_MAP}.items() for q in v}

# 🔴 `getattr(x,'a', x.b)`는 기본값을 먼저 평가한다 → 폴백을 쓰기도 전에 죽는다. 지연 평가(or)로.
_POST = (getattr(proc_d, 'post_process_grounded_object_detection', None)
         or getattr(proc_d, 'post_process_object_detection', None))
assert _POST, f'★후처리 API를 못 찾음: {[a for a in dir(proc_d) if "post_process" in a]}'

@torch.inference_mode()
def detect(paths, thr=0.02, topk=50, bs=8, drop_neg=True):
    jobs = []
    for p in paths:
        im = Image.open(p).convert('RGB')
        S = max(im.size)
        sq = Image.new('RGB', (S, S), (0, 0, 0)); sq.paste(im, (0, 0))
        jobs.append((p, sq, S))
    out = {p: [] for p in paths}
    for i in range(0, len(jobs), bs):
        blk = jobs[i:i + bs]
        inp = proc_d(text=[QUERIES] * len(blk), images=[j[1] for j in blk],
                     return_tensors='pt').to(DEV)
        o = det(**inp)
        sizes = torch.tensor([[j[2], j[2]] for j in blk], device=DEV)
        for (p, _s, _S), r in zip(blk, _POST(outputs=o, threshold=thr, target_sizes=sizes)):
            for b, s, l in zip(r['boxes'].tolist(), r['scores'].tolist(), r['labels'].tolist()):
                q = QUERIES[l]; tag = TAG_OF[q]
                if drop_neg and tag in DROP_TAGS: continue
                out[p].append((b[0], b[1], b[2], b[3], float(s), tag, q))
    for p in out:
        out[p].sort(key=lambda t: -t[4]); out[p] = out[p][:topk]
    return out

print(f'OWLv2 로드 완료 · {DEV} · 질의 {len(QUERIES)}개 (결함 {sum(len(v) for v in QUERY_MAP.values())} '
      f'+ 정상 {sum(len(v) for v in NEG_MAP.values())}) · 후처리 API {_POST.__name__}')
print(f'revision 고정: {REVISION or "🔴 미고정 — §2의 해시를 REVISION에 넣을 것"}')

In [ ]:
# == §4 🔴 1단 OWLv2 지연 측정 — 모델카드 미측정 항목 ==
# 배포 가능성을 가르는 수치다. 셀 1개 = 약 270프레임이므로 장당 지연 × 270이 셀당 비용.
# 배치 크기를 바꿔가며 재고, GPU 워밍업 후 측정한다(첫 호출은 커널 컴파일로 훨씬 느리다).
import time, statistics as st

FRAMES_PER_CELL = 270          # ★운영 가정. 실제 값으로 바꿀 것
N_WARM, N_MEAS  = 4, 24
BATCHES = [1, 4, 8, 16]

_p = [IMGDIR / n for n in IMGS[:max(BATCHES) * 2]]
_sz = Image.open(_p[0]).size
print(f'측정 이미지 {len(_p)}장 · 크기 {_sz[0]}x{_sz[1]} · 워밍업 {N_WARM}장\n')

detect(_p[:N_WARM], thr=0.5, bs=4)                 # 워밍업
if DEV == 'cuda': torch.cuda.synchronize()

LAT = {}
for bs in BATCHES:
    ts = []
    for i in range(0, N_MEAS, bs):
        blk = [_p[(i + j) % len(_p)] for j in range(bs)]
        if DEV == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        detect(blk, thr=0.02, topk=50, bs=bs)
        if DEV == 'cuda': torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) / bs)
    per = st.median(ts)
    LAT[bs] = {'per_image_s': round(per, 4),
               'imgs_per_s': round(1 / per, 2),
               'per_cell_s': round(per * FRAMES_PER_CELL, 1),
               'per_cell_min': round(per * FRAMES_PER_CELL / 60, 2)}
    print(f'batch {bs:>2d}   장당 {per*1000:7.1f} ms   초당 {1/per:5.2f}장   '
          f'셀당({FRAMES_PER_CELL}프레임) {per*FRAMES_PER_CELL:6.1f}초 = {per*FRAMES_PER_CELL/60:.2f}분')

best = min(LAT, key=lambda b: LAT[b]['per_image_s'])
LAT['_meta'] = {'device': ENV.get('gpu'), 'image_size': list(_sz),
                'frames_per_cell': FRAMES_PER_CELL, 'queries': len(QUERIES),
                'best_batch': best, 'n_measure': N_MEAS}
(MC / 'latency_owlv2.json').write_text(json.dumps(LAT, ensure_ascii=False, indent=1),
                                       encoding='utf-8')
print(f'\n최적 배치 {best} · 셀당 {LAT[best]["per_cell_min"]}분')
print(f'저장 {MC/"latency_owlv2.json"}')
print('\n▶ 읽는 법')
print('  · 셀당 4초 목표라면 전량 추론은 불가 — 값싼 프레임 프리필터가 필요하다')
print(f'  · 질의 {len(QUERIES)}개를 **이미지마다 다시 인코딩**한다 — 질의는 고정이므로'
      ' 임베딩을 캐시하면 텍스트 인코더 비용이 통째로 빠진다(최대 레버)')
print('  · TensorRT/FP16 전환 시 3~5배가 통상 범위(미측정)')

In [ ]:
# == §5 운영점 재현 확인 — 게이트가 스펙대로 동작하는가 ==
# 🔴 스펙에 적힌 thr 0.08 · N≥8 이 지금 코드에서 그대로 나오는지 확인한다.
#    (라벨이 있는 프레임에서만 채점 가능)
THR_GATE, N_GATE, THR_LOC = 0.08, 8, 0.12

_lab = [n for n in IMGS if n in GT]
print(f'라벨 있는 프레임 {len(_lab)}장 — 전량 검출 중 (배치 {locals().get("best", 8)})...')
_D = {p.name: v for p, v in detect([IMGDIR / n for n in _lab], thr=0.02, topk=50,
                                   bs=locals().get('best', 8)).items()}

rows = []
for thr in (0.05, 0.08, 0.12, 0.18):
    for N in (1, 2, 5, 8, 12):
        tp = fp = fn = 0
        for n in _lab:
            pred = sum(1 for b in _D[n] if b[4] >= thr) >= N
            truth = bool(GT[n]['points'])
            tp += pred and truth; fp += pred and not truth; fn += (not pred) and truth
        P, R = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        rows.append((2 * P * R / max(P + R, 1e-9), thr, N, P, R, fp, fn))
rows.sort(reverse=True)
print(f'\n{"thr":>5s} {"N":>3s} {"P":>7s} {"R":>7s} {"F1":>7s} {"오탐":>5s} {"놓침":>5s}')
for F, thr, N, P, R, fp, fn in rows[:8]:
    print(f'{thr:5.2f} {N:3d} {P:7.3f} {R:7.3f} {F:7.3f} {fp:5d} {fn:5d}' +
          ('   ← 스펙 운영점' if (thr, N) == (THR_GATE, N_GATE) else ''))

_spec = next(r for r in rows if (r[1], r[2]) == (THR_GATE, N_GATE))
SPEC = {'thr': THR_GATE, 'N': N_GATE, 'P': round(_spec[3], 3), 'R': round(_spec[4], 3),
        'F1': round(_spec[0], 3), 'fp': _spec[5], 'fn': _spec[6], 'n_frames': len(_lab),
        'expected': {'P': 0.876, 'R': 1.000, 'F1': 0.934}}
(MC / 'operating_point.json').write_text(json.dumps(SPEC, ensure_ascii=False, indent=1),
                                         encoding='utf-8')
print(f'\n스펙 운영점 재현: P {SPEC["P"]} / R {SPEC["R"]} / F1 {SPEC["F1"]}')
print(f'   문서 기록값:     P 0.876 / R 1.000 / F1 0.934')
_ok = abs(SPEC['F1'] - 0.934) <= 0.02 and SPEC['R'] >= 0.98
print('   ' + ('✅ 일치' if _ok else '🔴 불일치 — 질의어·전처리·라이브러리 버전이 바뀌었는지 확인할 것'))
print(f'저장 {MC/"operating_point.json"}')

In [ ]:
# == §6 (선택) 2단 Qwen2.5-VL 지연 재측정 — GPU 16GB 필요 ==
# 이미 1.1초/호출로 실측돼 있다. 재확인이 목적이며, 건너뛰어도 §7~§9는 돌아간다.
# 🔴 OWLv2와 동시에 올라가므로 메모리가 빠듯하면 런타임을 재시작하고 이 셀부터 돌릴 것.
RUN_QWEN = True          # ★건너뛰려면 False

if RUN_QWEN:
    !pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils
    import torch, time, statistics as st
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    from PIL import Image, ImageDraw

    _kw = {'revision': None}
    vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VLM_ID, torch_dtype=torch.bfloat16, device_map='auto')
    proc_v = AutoProcessor.from_pretrained(VLM_ID)

    PROMPT = ('왼쪽은 같은 셀의 다른 부위다. 오른쪽이 검사 대상이다.\n'
              '왼쪽에는 없고 오른쪽에만 있는 것을 찾아라.\n\n'
              '오른쪽에서 보이는 것을 고르라. 설명하지 말고 아래 세 줄만 답하라.\n\n'
              '종류  0 없음  1 녹·부식  2 벗겨짐  3 찢김·파손  4 긁힘  5 들뜸  6 오염·이물질\n'
              '크기  A 사진 절반 이상   B 사진의 1/4쯤   C 손톱만 함   D 점 하나\n\n'
              '종류: (숫자)\n크기: (알파벳)\n본것: (한 문장)')

    def _gen(img, prompt, max_new=120):
        msgs = [{'role': 'user', 'content': [{'type': 'image', 'image': img},
                                             {'type': 'text', 'text': prompt}]}]
        text = proc_v.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = proc_v(text=[text], images=[img], return_tensors='pt').to(vlm.device)
        with torch.inference_mode():
            g = vlm.generate(**inp, max_new_tokens=max_new, do_sample=False)
        return proc_v.batch_decode(g[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0]

    _im = Image.open(IMGDIR / IMGS[0]).convert('RGB')
    _c = _im.crop((0, 0, min(336, _im.width), min(336, _im.height))).resize((336, 336))
    _panel = Image.new('RGB', (336 * 2 + 12, 336), (255, 255, 255))
    _panel.paste(_c, (0, 0)); _panel.paste(_c, (348, 0))

    for _ in range(2): _gen(_panel, PROMPT)          # 워밍업
    ts = []
    for _ in range(12):
        t0 = time.perf_counter(); _out = _gen(_panel, PROMPT); ts.append(time.perf_counter() - t0)
    QL = {'per_call_s': round(st.median(ts), 3), 'min': round(min(ts), 3),
          'max': round(max(ts), 3), 'n': len(ts), 'panel_size': list(_panel.size),
          'dtype': 'bfloat16', 'max_new_tokens': 120, 'expected_s': 1.1,
          'sample_output': _out[:200]}
    (MC / 'latency_qwen.json').write_text(json.dumps(QL, ensure_ascii=False, indent=1),
                                          encoding='utf-8')
    print(f'Qwen2.5-VL  중앙값 {QL["per_call_s"]}초/호출  (min {QL["min"]} / max {QL["max"]})')
    print(f'   문서 기록값 1.1초 — ' +
          ('✅ 일치' if abs(QL['per_call_s'] - 1.1) < 0.5 else '⚠️ 차이 있음(GPU·이미지 크기 확인)'))
    print(f'   셀당 결함 후보 N개면 2단 비용 = {QL["per_call_s"]}N초')
    print(f'\n샘플 출력:\n{_out[:200]}')
    print(f'\n저장 {MC/"latency_qwen.json"}')
else:
    print('건너뜀 — 2단 지연은 1.1초/호출(기록값)을 그대로 쓴다')

In [ ]:
# == §7 🔴 골든 fixture 20장 선정 + 기대 출력 스냅샷 ==
# 모델카드 필수 항목. 라이브러리·가중치·전처리가 바뀌면 이 스냅샷이 깨져서 알려준다.
# 🔴 선정은 **결정론적**이어야 한다(seed 고정, 정렬 후 추출). 매번 다른 20장이면 회귀 검증이 안 된다.
import hashlib, random

N_DEF, N_CLEAN, SEED = 12, 8, 7
_d = sorted(n for n in IMGS if n in GT and GT[n]['points'])
_c = sorted(n for n in IMGS if n in GT and not GT[n]['points'])
rg = random.Random(SEED)
FIX = sorted(rg.sample(_d, min(N_DEF, len(_d))) + rg.sample(_c, min(N_CLEAN, len(_c))))
print(f'픽스처 {len(FIX)}장 (결함 {min(N_DEF,len(_d))} · 무결함 {min(N_CLEAN,len(_c))}) · seed {SEED}\n')

_R = {p.name: v for p, v in detect([IMGDIR / n for n in FIX], thr=0.02, topk=50,
                                   bs=locals().get('best', 8)).items()}
SNAP = {'_meta': {'seed': SEED, 'n': len(FIX), 'thr_gate': THR_GATE, 'n_gate': N_GATE,
                  'thr_loc': THR_LOC, 'owl_id': OWL_ID, 'revision': REVISION,
                  'transformers': ENV.get('transformers'), 'gpu': ENV.get('gpu')},
        'frames': {}}
for n in FIX:
    b = Image.open(IMGDIR / n)
    boxes = _R[n]
    SNAP['frames'][n] = {
        'sha256': hashlib.sha256((IMGDIR / n).read_bytes()).hexdigest()[:16],
        'size': list(b.size),
        'human_points': len(GT[n]['points']),
        'human_label': '결함' if GT[n]['points'] else '정상',
        'n_box': {str(t): sum(1 for x in boxes if x[4] >= t) for t in (0.05, 0.08, 0.12, 0.18)},
        'gate': '결함' if sum(1 for x in boxes if x[4] >= THR_GATE) >= N_GATE else '정상',
        'top5': [{'bbox': [round(v, 1) for v in x[:4]], 'score': round(x[4], 4),
                  'tag': x[5], 'query': x[6]}
                 for x in sorted(boxes, key=lambda z: -z[4])[:5]],
    }
(MC / 'golden_fixture.json').write_text(json.dumps(SNAP, ensure_ascii=False, indent=1),
                                        encoding='utf-8')

_agree = sum(1 for n in FIX if SNAP['frames'][n]['gate'] == SNAP['frames'][n]['human_label'])
print(f'{"파일":52s} {"사람":>5s} {"게이트":>6s} {"박스@0.08":>10s}')
for n in FIX:
    f = SNAP['frames'][n]
    print(f'{n[-52:]:52s} {f["human_label"]:>5s} {f["gate"]:>6s} {f["n_box"]["0.08"]:>10d}'
          + ('' if f['gate'] == f['human_label'] else '   ✗'))
print(f'\n게이트 일치 {_agree}/{len(FIX)}')
print(f'저장 {MC/"golden_fixture.json"}')
print('\n▶ 이 파일을 **git에 커밋**하고 모델카드에 경로를 적는다.')
print('  배포 전마다 §8을 돌려 스냅샷이 깨지지 않았는지 확인한다.')

In [ ]:
# == §8 fixture 회귀 검증 — 저장된 스냅샷과 지금 결과를 비교 ==
# 라이브러리 업그레이드·가중치 변경·전처리 수정 뒤에 이 셀만 돌리면 된다.
# 🔴 부동소수 비차이는 허용하되 **게이트 판정과 박스 개수는 정확히 일치**해야 한다.
TOL_SCORE, TOL_BBOX = 2e-3, 1.0

SNAP0 = json.loads((MC / 'golden_fixture.json').read_text(encoding='utf-8'))
FIX0 = sorted(SNAP0['frames'])
print(f'스냅샷 {len(FIX0)}장 · 기록 환경 transformers={SNAP0["_meta"].get("transformers")} '
      f'/ gpu={SNAP0["_meta"].get("gpu")}')
print(f'현재 환경           transformers={ENV.get("transformers")} / gpu={ENV.get("gpu")}\n')

_R2 = {p.name: v for p, v in detect([IMGDIR / n for n in FIX0], thr=0.02, topk=50,
                                    bs=locals().get('best', 8)).items()}
bad = []
for n in FIX0:
    e = SNAP0['frames'][n]; boxes = sorted(_R2[n], key=lambda z: -z[4])
    if hashlib.sha256((IMGDIR / n).read_bytes()).hexdigest()[:16] != e['sha256']:
        bad.append((n, '이미지 파일이 바뀌었다')); continue
    g = '결함' if sum(1 for x in boxes if x[4] >= SNAP0['_meta']['thr_gate']) >= SNAP0['_meta']['n_gate'] else '정상'
    if g != e['gate']: bad.append((n, f'게이트 {e["gate"]} → {g}'))
    for t, v in e['n_box'].items():
        c = sum(1 for x in boxes if x[4] >= float(t))
        if c != v: bad.append((n, f'박스수@{t} {v} → {c}'))
    for k, ex in enumerate(e['top5']):
        if k >= len(boxes): bad.append((n, f'top{k+1} 없음')); continue
        g2 = boxes[k]
        if g2[5] != ex['tag']: bad.append((n, f'top{k+1} tag {ex["tag"]} → {g2[5]}'))
        if abs(g2[4] - ex['score']) > TOL_SCORE:
            bad.append((n, f'top{k+1} score {ex["score"]} → {round(g2[4],4)}'))
        if max(abs(a - b) for a, b in zip(g2[:4], ex['bbox'])) > TOL_BBOX:
            bad.append((n, f'top{k+1} bbox 이동 > {TOL_BBOX}px'))

if bad:
    print(f'🔴 회귀 {len(bad)}건')
    for n, why in bad[:25]: print(f'   {n[-40:]:40s} {why}')
    if len(bad) > 25: print(f'   ... 외 {len(bad)-25}건')
    print('\n원인 후보: 라이브러리 버전 · 모델 revision · 전처리(정사각 패딩) · 질의어 목록 · GPU/dtype')
else:
    print(f'✅ 회귀 없음 — {len(FIX0)}장 전부 스냅샷과 일치 '
          f'(score 허용오차 {TOL_SCORE} · bbox {TOL_BBOX}px)')

In [ ]:
# == §9 모델카드 붙여넣기용 요약 ==
# 여기 출력을 그대로 모델카드에 붙인다. 미확정 항목은 🔴로 남는다.
def _load(f):
    p = MC / f
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None

E, R, L, Q, S, G = (_load(x) for x in ('env.json', 'revisions.json', 'latency_owlv2.json',
                                       'latency_qwen.json', 'operating_point.json',
                                       'golden_fixture.json'))
L2 = {k: v for k, v in (L or {}).items() if k != '_meta'}
bb = (L or {}).get('_meta', {}).get('best_batch')

lines = ['## 환경', '',
         f'- python `{(E or {}).get("python")}` · {(E or {}).get("platform")}',
         f'- GPU `{(E or {}).get("gpu")}` {(E or {}).get("gpu_mem_GB")}GB · '
         f'driver `{(E or {}).get("nvidia_driver")}` · CUDA `{(E or {}).get("torch.version.cuda")}`', '',
         '| 패키지 | 버전 |', '|---|---|']
for k in ('transformers', 'tokenizers', 'huggingface-hub', 'accelerate', 'torch',
          'torchvision', 'pillow', 'numpy', 'qwen-vl-utils'):
    v = (E or {}).get(k)
    lines.append(f'| `{k}` | {"`"+str(v)+"`" if v else "🔴 미설치"} |')

lines += ['', '## 모델 식별', '', '| 모델 | model_id | revision (sha) |', '|---|---|---|']
for rid in (OWL_ID, VLM_ID):
    r = (R or {}).get(rid, {})
    loc = (r.get('local_snapshots') or [None])[0]      # 실제로 받은 것 = 우선
    rem = r.get('sha')                                  # 원격 main = 폴백
    cell = (f'`{loc}` (로컬)' if loc else
            (f'`{rem}` (원격 main · 🔴 로컬 미확인, §3 뒤 §2 재실행 권장)' if rem else '🔴 미확인'))
    lines.append(f'| {"검출" if rid==OWL_ID else "해설"} | `{rid}` | {cell} |')

lines += ['', '## 지연 (실측)', '']
if L2:
    lines += ['| 배치 | 장당 | 초당 | 셀당(270프레임) |', '|---|---|---|---|']
    for b, v in sorted(L2.items(), key=lambda kv: int(kv[0])):
        lines.append(f'| {b} | {v["per_image_s"]*1000:.0f} ms | {v["imgs_per_s"]}장 | '
                     f'{v["per_cell_min"]}분 |')
    lines.append(f'\n최적 배치 **{bb}**')
else:
    lines.append('🔴 1단 지연 미측정 — §4를 돌릴 것')
if Q: lines.append(f'\n2단 Qwen2.5-VL **{Q["per_call_s"]}초/호출** (bf16, 패널 '
                   f'{Q["panel_size"][0]}x{Q["panel_size"][1]}, max_new_tokens {Q["max_new_tokens"]})')
else: lines.append('\n2단 지연: 기록값 1.1초/호출 (§6 미실행)')

lines += ['', '## 운영점', '']
if S:
    lines += [f'- 게이트 `thr={S["thr"]}` · 박스 `{S["N"]}개 이상` → 결함',
              f'- **P {S["P"]} / R {S["R"]} / F1 {S["F1"]}** (프레임 {S["n_frames"]}장, '
              f'오탐 {S["fp"]} · 놓침 {S["fn"]})',
              f'- 위치 특정 `thr=0.12` · 상위 6박스']
else: lines.append('🔴 §5 미실행')

lines += ['', '## 골든 fixture', '']
if G:
    m = G['_meta']
    lines += [f'- `{MC}/golden_fixture.json` · {m["n"]}장 (seed {m["seed"]})',
              f'- 기록 환경: transformers `{m.get("transformers")}` / gpu `{m.get("gpu")}`',
              f'- 검증: 이 노트북 §8 (게이트·박스수 정확 일치, score 2e-3 · bbox 1px 허용)']
else: lines.append('🔴 §7 미실행')

lines += ['', '## 🔴 남은 미확정', '']
todo = []
for _rid, _nm in ((OWL_ID, 'OWLv2'), (VLM_ID, 'Qwen')):
    _r = (R or {}).get(_rid, {})
    if not _r.get('local_snapshots'):
        todo.append(f'{_nm} revision — 로컬 스냅샷 미확인'
                    + (' (원격 sha는 확보, §3 뒤 §2 재실행하면 확정)' if _r.get('sha') else ''))
if not (E or {}).get('transformers'): todo.append('transformers 버전')
if not L2: todo.append('1단 OWLv2 지연')
if not G: todo.append('골든 fixture')
todo.append('계약 enum 확장 협의 (`CORROSION`·`PEELING`) — 코드로 확인 불가, 팀 협의 항목')
lines += [f'- {x}' for x in todo]

TXT = '\n'.join(lines)
(MC / 'modelcard_summary.md').write_text(TXT, encoding='utf-8')
print(TXT)
print(f'\n{"="*60}\n저장 {MC/"modelcard_summary.md"}')
print(f'산출물 전체: {sorted(p.name for p in MC.iterdir())}')

In [ ]:
# == §10 🔴 배포 픽스처 회귀 — GPU에서 1회 (카드 §7.4) ==
# 선행: §0 (OUT·MC 경로). GPU 필요. OWLv2 로드 ~1분 + 20장 추론 ~10초.
#
# 🔑 §8 과 다른 것을 검사한다
#     §8  = 평가 픽스처 — 이미 잘린 크롭 입력 · thr 0.08 · 검출기만
#     §10 = 배포 픽스처 — **원본 1920x1080 입력** · thr 0.10 · **크롭까지 포함한 배포 경로 전체**
#   JPEG 압축 세대가 검출 수를 53% 바꾸므로 둘은 바꿔 쓸 수 없다. 잘못 넣으면 도구가 막는다.
import sys, json, zipfile, importlib
from pathlib import Path

def _grab(label, cands, patt, is_dir=False):
    """Drive·/content 에서 찾고, 없으면 업로드받는다."""
    for c in cands:
        c = Path(c)
        if (c.is_dir() if is_dir else c.is_file()):
            print(f'  {label:22s} {c}')
            return c
    print(f'  {label:22s} ★ 못 찾음 — {patt} 를 업로드하세요')
    from google.colab import files
    up = files.upload()
    got = [Path.cwd() / n for n in up]          # files.upload() 는 cwd 에 쓴다
    for g in got:                                  # zip 이면 풀어서 폴더를 돌려준다
        if g.suffix == '.zip':
            d = Path.cwd() / g.stem
            zipfile.ZipFile(g).extractall(d)
            inner = [x for x in d.rglob('*') if x.is_dir() and list(x.glob('*.jpg'))]
            d = inner[0] if inner else d
            print(f'  {"":22s} → 압축 해제 {d}')
            return d
    return got[0].parent if is_dir else got[0]

print('■ 파일 찾기')
MOD = _grab('ext_infer.py', [
    '/content/ext_infer.py', MC / 'ext_infer.py', OUT / 'ext_infer.py',
    '/content/drive/MyDrive/ext_infer.py',
    '/content/kt-aivle-big-proj-model-rgb/ext_infer.py'], 'ext_infer.py')
FX = _grab('픽스처 JSON', [
    MC / 'golden_fixture_deploy.json', '/content/golden_fixture_deploy.json',
    OUT / 'golden_fixture_deploy.json'], 'golden_fixture_deploy.json')
IMD = _grab('원본 20장 폴더', [
    MC / 'fixtures_deploy', '/content/fixtures_deploy',
    OUT / 'fixtures_deploy'], 'fixtures_deploy 폴더 또는 zip', is_dir=True)

# ── 모듈 적재 (재실행 시 갱신되도록 reload) ──
sys.path.insert(0, str(Path(MOD).parent))
import ext_infer; importlib.reload(ext_infer)

import torch
if not torch.cuda.is_available():
    print('\n🔴 GPU 가 없다 — 런타임 유형을 GPU 로 바꿀 것. CPU 로는 의미 있는 검사가 안 된다')

cfg = ext_infer.Config()
print(f'\n■ 현재 설정  thr_gate {cfg.thr_gate} · n_gate {cfg.n_gate} · revision {cfg.owl_rev[:12]}…')
print(f'  픽스처 이미지 {len(list(Path(IMD).glob("*.jpg")))}장\n')

r = ext_infer.verify_fixture(FX, IMD, cfg)

# ── 결과를 모델카드 산출물로 남긴다 ──
(MC / 'fixture_verify_deploy.json').write_text(json.dumps({
    'n': r['n'], 'fail': [list(x) for x in r['fail']], 'warn': [list(x) for x in r['warn']],
    'thr_gate': cfg.thr_gate, 'revision': cfg.owl_rev,
    'transformers': __import__('transformers').__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, ensure_ascii=False, indent=1), encoding='utf-8')

print(f'\n기록 → {MC / "fixture_verify_deploy.json"}')
if not r['fail'] and r['n'] == 20:
    print('✅ 카드 §7.4 · 스펙 §9.6 의 🔴 "GPU 1회 실행" 을 닫아도 된다')
elif r['fail']:
    print('🔴 배포하지 말 것 — 원인 후보: 라이브러리 버전 · 모델 revision · 크롭 파라미터 · 입력 재인코딩')
else:
    print(f'⚠️ {r["n"]}/20 장만 검사됐다 — 이미지 폴더를 확인할 것')
